<a href="https://colab.research.google.com/github/manikanta-eng/HPC/blob/main/hpc_lab7_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Assignment 1  smid vector addition**

In [2]:
import numpy as np
import time
from numba import njit, prange


# -----------------------------
# Pure Python Loop
# -----------------------------
def python_add(a, b):
    c = np.zeros_like(a)
    for i in range(len(a)):
        c[i] = a[i] + b[i]
    return c


# -----------------------------
# Numba SIMD (Single Thread)
# -----------------------------
@njit(fastmath=True)
def numba_simd_add(a, b):
    c = np.empty_like(a)
    for i in range(a.size):
        c[i] = a[i] + b[i]
    return c


# -----------------------------
# Numba Parallel + SIMD
# -----------------------------
@njit(parallel=True, fastmath=True)
def numba_parallel_add(a, b):
    c = np.empty_like(a)
    for i in prange(a.size):
        c[i] = a[i] + b[i]
    return c


# -----------------------------
# Timing Helper
# -----------------------------
def time_it(func, *args):
    start = time.time()
    out = func(*args)
    end = time.time()
    return out, end - start


def main():
    N = 10_000_000

    print("Initializing data...")
    A = np.random.rand(N)
    B = np.random.rand(N)

    # Warm-up (important for Numba JIT compilation)
    numba_simd_add(A[:10], B[:10])
    numba_parallel_add(A[:10], B[:10])

    print("\n==== PERFORMANCE COMPARISON =====")

    _, t_python = time_it(python_add, A, B)
    print(f"Python loop time        : {t_python:.4f} s")

    # NumPy vectorized
    start = time.time()
    C_numpy = A + B
    t_numpy = time.time() - start
    print(f"NumPy vectorized time   : {t_numpy:.4f} s")

    _, t_simd = time_it(numba_simd_add, A, B)
    print(f"Numba SIMD time         : {t_simd:.4f} s")

    _, t_parallel = time_it(numba_parallel_add, A, B)
    print(f"Numba Parallel time     : {t_parallel:.4f} s")

    # Bandwidth estimation
    bytes_moved = A.nbytes * 3  # read A + read B + write C
    bandwidth = bytes_moved / t_numpy / 1e9

    print("\n==== BANDWIDTH =====")
    print(f"Data moved              : {bytes_moved / 1e9:.2f} GB")
    print(f"Estimated bandwidth     : {bandwidth:.2f} GB/s")


if __name__ == "__main__":
    main()

Initializing data...

==== PERFORMANCE COMPARISON =====
Python loop time        : 2.1833 s
NumPy vectorized time   : 0.0167 s
Numba SIMD time         : 0.0304 s
Numba Parallel time     : 0.0207 s

==== BANDWIDTH =====
Data moved              : 0.24 GB
Estimated bandwidth     : 14.39 GB/s


**assignment 2**

In [7]:
import numpy as np
import time
from numba import njit


# -----------------------------------
# 1. Initialize large array
# -----------------------------------
N = 50_000_000  # 50 million elements
print("Initializing array...")
arr = np.random.rand(N).astype(np.float64)


# -----------------------------------
# 2. Normal Python Loop Sum
# -----------------------------------
def normal_sum(a):
    total = 0.0
    for i in range(len(a)):
        total += a[i]
    return total


# -----------------------------------
# 3. SIMD Reduction using Numba
# -----------------------------------
@njit(fastmath=True)   # enables SIMD vectorization
def simd_sum(a):
    total = 0.0
    for i in range(a.size):
        total += a[i]     # reduction happens automatically
    return total


# -----------------------------------
# Warm-up (JIT compilation)
# -----------------------------------
simd_sum(arr[:10])


# -----------------------------------
# Timing Normal Loop
# -----------------------------------
start = time.time()
sum_normal = normal_sum(arr)
time_normal = time.time() - start

print("\nNormal Loop Sum  :", sum_normal)
print("Normal Loop Time :", round(time_normal, 4), "seconds")


# -----------------------------------
# Timing SIMD Reduction
# -----------------------------------
start = time.time()
sum_simd = simd_sum(arr)
time_simd = time.time() - start

print("\nSIMD Reduction Sum  :", sum_simd)
print("SIMD Reduction Time :", round(time_simd, 4), "seconds")


# -----------------------------------
# 4. Verify Correctness
# -----------------------------------
if np.allclose(sum_normal, sum_simd):
    print("\nResult Verified: Both sums match ✅")
else:
    print("\nMismatch in results ❌")


# -----------------------------------
# Performance Comparison
# -----------------------------------
speedup = time_normal / time_simd
print("\nSpeedup:", round(speedup, 2), "x faster")

Initializing array...

Normal Loop Sum  : 24997007.74063507
Normal Loop Time : 10.2793 seconds

SIMD Reduction Sum  : 24997007.740632527
SIMD Reduction Time : 0.0298 seconds

Result Verified: Both sums match ✅

Speedup: 344.9 x faster


**Assignment 3**

In [5]:
import numpy as np
import time
from numba import njit


@njit(fastmath=True)
def simd_add(a, b):
    c = np.empty_like(a)
    for i in range(a.size):
        c[i] = a[i] + b[i]
    return c


def main():
    N = 50_000_000
    print("Allocating arrays...")

    A_aligned = np.random.rand(N).astype(np.float64)
    B_aligned = np.random.rand(N).astype(np.float64)

    A_unaligned = A_aligned[1:]
    B_unaligned = B_aligned[1:]

    simd_add(A_aligned[:10], B_aligned[:10])

    print("\n==== SIMD Alignment Experiment ====")

    start = time.time()
    simd_add(A_aligned, B_aligned)
    time_aligned = time.time() - start
    print(f"Aligned Time   : {time_aligned:.4f} seconds")

    start = time.time()
    simd_add(A_unaligned, B_unaligned)
    time_unaligned = time.time() - start
    print(f"Unaligned Time : {time_unaligned:.4f} seconds")

    slowdown = time_unaligned / time_aligned
    print(f"\nUnaligned is {slowdown:.2f}x slower than aligned")


if __name__ == "__main__":
    main()

Allocating arrays...

==== SIMD Alignment Experiment ====
Aligned Time   : 1.2462 seconds
Unaligned Time : 0.1829 seconds

Unaligned is 0.15x slower than aligned


**Assignment 4**

In [8]:
import numpy as np
import time
from numba import njit, prange


@njit(parallel=True)
def parallel_add(a, b, c):
    for i in prange(a.size):
        c[i] = a[i] + b[i]


@njit(parallel=True, fastmath=True)
def parallel_simd_add(a, b, c):
    for i in prange(a.size):
        c[i] = a[i] + b[i]


def main():
    N = 50_000_000

    A = np.random.rand(N).astype(np.float64)
    B = np.random.rand(N).astype(np.float64)
    C = np.empty_like(A)

    parallel_add(A[:10], B[:10], C[:10])
    parallel_simd_add(A[:10], B[:10], C[:10])

    start = time.time()
    parallel_add(A, B, C)
    t_parallel = time.time() - start

    start = time.time()
    parallel_simd_add(A, B, C)
    t_parallel_simd = time.time() - start

    print("Parallel time        :", round(t_parallel, 4), "seconds")
    print("Parallel + SIMD time :", round(t_parallel_simd, 4), "seconds")
    print("Speedup              :", round(t_parallel / t_parallel_simd, 2), "x")


if __name__ == "__main__":
    main()

Parallel time        : 0.0691 seconds
Parallel + SIMD time : 0.0529 seconds
Speedup              : 1.31 x


**Assignment 5**

In [9]:
import numpy as np
import time
from numba import njit


@njit(fastmath=True)
def branch_loop(a, b, c):
    for i in range(a.size):
        if a[i] > 0.5:
            c[i] = a[i] * b[i]
        else:
            c[i] = a[i] + b[i]


@njit(fastmath=True)
def simd_friendly_loop(a, b, c):
    for i in range(a.size):
        mask = a[i] > 0.5
        mult = a[i] * b[i]
        add = a[i] + b[i]
        c[i] = mult * mask + add * (1.0 - mask)


def main():
    N = 50_000_000

    A = np.random.rand(N).astype(np.float64)
    B = np.random.rand(N).astype(np.float64)
    C = np.empty_like(A)

    branch_loop(A[:10], B[:10], C[:10])
    simd_friendly_loop(A[:10], B[:10], C[:10])

    start = time.time()
    branch_loop(A, B, C)
    t_branch = time.time() - start

    start = time.time()
    simd_friendly_loop(A, B, C)
    t_simd = time.time() - start

    print("Branch loop time       :", round(t_branch, 4), "seconds")
    print("SIMD-friendly time     :", round(t_simd, 4), "seconds")
    print("Speedup                :", round(t_branch / t_simd, 2), "x")


if __name__ == "__main__":
    main()

Branch loop time       : 0.0743 seconds
SIMD-friendly time     : 0.0542 seconds
Speedup                : 1.37 x
